**RESNET-18**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import numpy as np
import pandas as pd
import os
import gc
import timm
import copy
import matplotlib.pyplot as plt

# --- AYARLAR ---
BASE_PATH = '/content/drive/MyDrive/Deep-Learning'
BATCH_SIZE = 256
EPOCHS = 12
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'resnet18'

# --- DATASET ---
class PureImageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = torch.LongTensor(labels)
        self.transform = transform
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def get_transforms():
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# --- MODEL ---
class SingleModalityImageModel(nn.Module):
    def __init__(self, model_name, num_classes=9):
        super(SingleModalityImageModel, self).__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    def forward(self, x): return self.backbone(x)

# --- ÇALIŞTIRMA ---
def run_resnet():
    print(f"🚀 {MODEL_NAME.upper()} EĞİTİMİ BAŞLIYOR (Mixed Precision)...")

    X_img = np.load(os.path.join(BASE_PATH, 'X_images_uint8.npy'), mmap_mode='r')
    y = np.load(os.path.join(BASE_PATH, 'y_labels.npy'))

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_accs, fold_f1s = [], []

    # Hızlandırıcı (Colab PRO için)
    scaler = torch.cuda.amp.GradScaler()

    for fold, (train_val_idx, test_idx) in enumerate(skf.split(X_img, y)):
        print(f"\n🔄 FOLD {fold+1}/{N_FOLDS}...")
        inner_train_idx, inner_val_idx = train_test_split(train_val_idx, test_size=0.1875, stratify=y[train_val_idx], random_state=42)

        train_ds = PureImageDataset(X_img[inner_train_idx], y[inner_train_idx], transform=get_transforms())
        val_ds = PureImageDataset(X_img[inner_val_idx], y[inner_val_idx], transform=get_transforms())
        test_ds = PureImageDataset(X_img[test_idx], y[test_idx], transform=get_transforms())

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

        model = SingleModalityImageModel(MODEL_NAME).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.CrossEntropyLoss()

        best_acc = 0.0; best_wts = copy.deepcopy(model.state_dict())

        for ep in range(EPOCHS):
            model.train()
            for i, l in tqdm(train_loader, desc=f"Ep {ep+1}", leave=False):
                i, l = i.to(DEVICE), l.to(DEVICE)
                optimizer.zero_grad()

                # Mixed Precision
                with torch.cuda.amp.autocast():
                    out = model(i)
                    loss = criterion(out, l)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            # Validation
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for i, l in val_loader:
                    i, l = i.to(DEVICE), l.to(DEVICE)
                    out = model(i)
                    _, p = torch.max(out, 1)
                    total += l.size(0); correct += (p == l).sum().item()

            val_acc = correct / total
            if val_acc > best_acc: best_acc = val_acc; best_wts = copy.deepcopy(model.state_dict())

        # TEST
        model.load_state_dict(best_wts); model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for i, l in test_loader:
                _, p = torch.max(model(i.to(DEVICE)), 1)
                preds.extend(p.cpu().numpy()); trues.extend(l.cpu().numpy())

        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average='macro')
        fold_accs.append(acc); fold_f1s.append(f1)
        print(f"   -> Fold {fold+1} Test Acc: {acc:.4f} | F1: {f1:.4f}")

        del model, train_loader, val_loader, test_loader; gc.collect(); torch.cuda.empty_cache()

    # --- TABLO ---
    print("\n" + "="*85)
    print(f"🏆 {MODEL_NAME.upper()} SONUÇ TABLOSU (Kılavuza Uygun)")
    print("="*85)
    mean_acc, std_acc = np.mean(fold_accs), np.std(fold_accs)
    mean_f1, std_f1 = np.mean(fold_f1s), np.std(fold_f1s)

    results = {
        'Metric': ['Accuracy', 'F1-Score (Macro)'],
        'Fold1': [fold_accs[0], fold_f1s[0]], 'Fold2': [fold_accs[1], fold_f1s[1]],
        'Fold3': [fold_accs[2], fold_f1s[2]], 'Fold4': [fold_accs[3], fold_f1s[3]], 'Fold5': [fold_accs[4], fold_f1s[4]],
        'Mean ± Std': [f"{mean_acc:.4f} ± {std_acc:.4f}", f"{mean_f1:.4f} ± {std_f1:.4f}"]
    }
    print(pd.DataFrame(results).to_string(index=False))
    print("="*85)

if __name__ == "__main__":
    run_resnet()

🚀 RESNET18 EĞİTİMİ BAŞLIYOR (Mixed Precision)...


/tmp/ipython-input-267194951.py:63: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



🔄 FOLD 1/5...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

...

   -> Fold 3 Test Acc: 0.8274 | F1: 0.7890

🔄 FOLD 4/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-267194951.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Fold 4 Test Acc: 0.8149 | F1: 0.7708

🔄 FOLD 5/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-267194951.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Fold 5 Test Acc: 0.8101 | F1: 0.7670

🏆 RESNET18 SONUÇ TABLOSU (Kılavuza Uygun)
          Metric    Fold1    Fold2    Fold3    Fold4    Fold5      Mean ± Std
        Accuracy 0.813603 0.808149 0.827398 0.814886 0.810074 0.8148 ± 0.0067
F1-Score (Macro) 0.771548 0.768220 0.789008 0.770785 0.766969 0.7733 ± 0.0080


**MLP**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd
import os
import copy

BASE_PATH = '/content/drive/MyDrive/Deep-Learning'
BATCH_SIZE = 256
EPOCHS = 20
N_FOLDS = 5
DEVICE = 'cuda'

class TabularDataset(Dataset):
    def __init__(self, stats, labels):
        self.stats = torch.FloatTensor(stats)
        self.labels = torch.LongTensor(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.stats[idx], self.labels[idx]

class MLPModel(nn.Module):
    def __init__(self, input_dim=27, num_classes=9):
        super(MLPModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    def forward(self, x): return self.net(x)

def run_mlp():
    MODEL_NAME = "MLP (Tabular)"
    print(f"🚀 {MODEL_NAME} EĞİTİMİ BAŞLIYOR...")

    X_stats = np.load(os.path.join(BASE_PATH, 'X_stats.npy'))
    y = np.load(os.path.join(BASE_PATH, 'y_labels.npy'))
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_accs, fold_f1s = [], []

    for fold, (train_val_idx, test_idx) in enumerate(skf.split(X_stats, y)):
        print(f"🔄 FOLD {fold+1}/{N_FOLDS}...", end=" ")
        inner_train_idx, inner_val_idx = train_test_split(train_val_idx, test_size=0.1875, stratify=y[train_val_idx], random_state=42)

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_stats[inner_train_idx])
        X_val = scaler.transform(X_stats[inner_val_idx])
        X_test = scaler.transform(X_stats[test_idx])

        train_loader = DataLoader(TabularDataset(X_train, y[inner_train_idx]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(TabularDataset(X_val, y[inner_val_idx]), batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(TabularDataset(X_test, y[test_idx]), batch_size=BATCH_SIZE, shuffle=False)

        model = MLPModel().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        best_acc = 0.0; best_wts = copy.deepcopy(model.state_dict())

        for ep in range(EPOCHS):
            model.train()
            for s, l in train_loader:
                optimizer.zero_grad(); loss = criterion(model(s.to(DEVICE)), l.to(DEVICE)); loss.backward(); optimizer.step()

            model.eval(); correct, total = 0, 0
            with torch.no_grad():
                for s, l in val_loader:
                    out = model(s.to(DEVICE)); _, p = torch.max(out, 1); total+=l.size(0); correct+=(p==l.to(DEVICE)).sum().item()
            if (correct/total) > best_acc: best_acc = correct/total; best_wts = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_wts); model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for s, l in test_loader:
                _, p = torch.max(model(s.to(DEVICE)), 1); preds.extend(p.cpu().numpy()); trues.extend(l.cpu().numpy())

        acc = accuracy_score(trues, preds); f1 = f1_score(trues, preds, average='macro')
        fold_accs.append(acc); fold_f1s.append(f1)
        print(f"Bitti. Test Acc: {acc:.4f}")

    print("\n" + "="*85)
    print(f"🏆 {MODEL_NAME} SONUÇ TABLOSU")
    print("="*85)
    res = {
        'Metric': ['Accuracy', 'F1-Score (Macro)'],
        'Mean ± Std': [f"{np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}", f"{np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}"]
    }
    print(pd.DataFrame(res).to_string(index=False))

if __name__ == "__main__":
    run_mlp()

🚀 MLP (Tabular) EĞİTİMİ BAŞLIYOR...
🔄 FOLD 1/5... Bitti. Test Acc: 0.6756
🔄 FOLD 2/5... Bitti. Test Acc: 0.6789
🔄 FOLD 3/5... Bitti. Test Acc: 0.6840
🔄 FOLD 4/5... Bitti. Test Acc: 0.6679
🔄 FOLD 5/5... Bitti. Test Acc: 0.6756

🏆 MLP (Tabular) SONUÇ TABLOSU
          Metric      Mean ± Std
        Accuracy 0.6764 ± 0.0052
F1-Score (Macro) 0.5625 ± 0.0058


**EARLY FUSION**

In [ ]:
# --- EARLY FUSION ---
MODEL_NAME = "Early Fusion"

class EarlyFusionModel(nn.Module):
    def __init__(self, num_classes=9):
        super(EarlyFusionModel, self).__init__()
        self.img_net = timm.create_model('resnet18', pretrained=True, num_classes=0) # 512 çıkış
        self.tab_net = nn.Sequential(nn.Linear(27, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(512+64, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, i, t):
        f_i = self.img_net(i)
        f_t = self.tab_net(t)
        return self.head(torch.cat((f_i, f_t), dim=1))

# Fusion Dataset
class FusionDataset(Dataset):
    def __init__(self, images, stats, labels, transform=None):
        self.images = images; self.stats = torch.FloatTensor(stats); self.labels = torch.LongTensor(labels); self.transform = transform
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform: img = self.transform(img)
        return img, self.stats[idx], self.labels[idx]

def run_early_fusion():
    print(f"🚀 {MODEL_NAME.upper()} BAŞLIYOR...")
    X_img = np.load(os.path.join(BASE_PATH, 'X_images_uint8.npy'), mmap_mode='r')
    X_stats = np.load(os.path.join(BASE_PATH, 'X_stats.npy'))
    y = np.load(os.path.join(BASE_PATH, 'y_labels.npy'))

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_accs, fold_f1s = [], []
    scaler_amp = torch.cuda.amp.GradScaler()

    for fold, (train_val_idx, test_idx) in enumerate(skf.split(X_img, y)):
        print(f"\n🔄 FOLD {fold+1}/{N_FOLDS}...")
        inner_train_idx, inner_val_idx = train_test_split(train_val_idx, test_size=0.1875, stratify=y[train_val_idx], random_state=42)

        s_scaler = StandardScaler()
        S_tr = s_scaler.fit_transform(X_stats[inner_train_idx])
        S_va = s_scaler.transform(X_stats[inner_val_idx])
        S_te = s_scaler.transform(X_stats[test_idx])

        train_l = DataLoader(FusionDataset(X_img[inner_train_idx], S_tr, y[inner_train_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
        val_l = DataLoader(FusionDataset(X_img[inner_val_idx], S_va, y[inner_val_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
        test_l = DataLoader(FusionDataset(X_img[test_idx], S_te, y[test_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

        model = EarlyFusionModel().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.CrossEntropyLoss()
        best_acc = 0.0; best_wts = copy.deepcopy(model.state_dict())

        for ep in range(EPOCHS):
            model.train()
            for i, s, l in tqdm(train_l, desc=f"Ep {ep+1}", leave=False):
                i, s, l = i.to(DEVICE), s.to(DEVICE), l.to(DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    out = model(i, s)
                    loss = criterion(out, l)
                scaler_amp.scale(loss).backward()
                scaler_amp.step(optimizer)
                scaler_amp.update()

            model.eval(); corr=0; tot=0
            with torch.no_grad():
                for i, s, l in val_l:
                    out = model(i.to(DEVICE), s.to(DEVICE))
                    _, p = torch.max(out, 1); corr+=(p==l.to(DEVICE)).sum().item(); tot+=l.size(0)
            if (corr/tot) > best_acc: best_acc=corr/tot; best_wts=copy.deepcopy(model.state_dict())

        model.load_state_dict(best_wts); model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for i, s, l in test_l:
                _, p = torch.max(model(i.to(DEVICE), s.to(DEVICE)), 1)
                preds.extend(p.cpu().numpy()); trues.extend(l.cpu().numpy())

        acc = accuracy_score(trues, preds); f1 = f1_score(trues, preds, average='macro')
        fold_accs.append(acc); fold_f1s.append(f1)
        print(f"   -> Test Acc: {acc:.4f} | F1: {f1:.4f}")
        del model, train_l, val_l, test_l; gc.collect(); torch.cuda.empty_cache()

    print("\n" + "="*85)
    print(f"🏆 {MODEL_NAME} SONUÇ TABLOSU")
    res = {'Metric': ['Accuracy', 'F1-Score'], 'Mean ± Std': [f"{np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}", f"{np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}"]}
    print(pd.DataFrame(res).to_string(index=False))

if __name__ == "__main__":
    run_early_fusion()

🚀 EARLY FUSION BAŞLIYOR...

🔄 FOLD 1/5...


/tmp/ipython-input-1629578985.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler()
Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-1629578985.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8200 | F1: 0.7686

🔄 FOLD 2/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-1629578985.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8197 | F1: 0.7742

🔄 FOLD 3/5...


...

   -> Test Acc: 0.8341 | F1: 0.7913

🔄 FOLD 4/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-1629578985.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8126 | F1: 0.7604

🔄 FOLD 5/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-1629578985.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8171 | F1: 0.7705

🏆 Early Fusion SONUÇ TABLOSU
  Metric      Mean ± Std
Accuracy 0.8207 ± 0.0072
F1-Score 0.7730 ± 0.0102


**INTERMEDIATE FUSION**

In [ ]:
# --- INTERMEDIATE FUSION ---
MODEL_NAME = "Intermediate Fusion"

class InterFusionModel(nn.Module):
    def __init__(self, num_classes=9):
        super(InterFusionModel, self).__init__()
        self.img_net = timm.create_model('resnet18', pretrained=True, features_only=True, out_indices=[4])
        self.tab_net = nn.Sequential(nn.Linear(27, 128), nn.ReLU(), nn.Linear(128, 512))
        self.attn = nn.MultiheadAttention(512, 8, batch_first=True)
        self.head = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, num_classes))
    def forward(self, i, t):
        f_i = self.img_net(i)[0].flatten(2).transpose(1, 2)
        f_t = self.tab_net(t).unsqueeze(1)
        attn_out, _ = self.attn(query=f_t, key=f_i, value=f_i)
        return self.head((attn_out + f_t).squeeze(1))

def run_inter_fusion():
    print(f"🚀 {MODEL_NAME.upper()} BAŞLIYOR...")
    X_img = np.load(os.path.join(BASE_PATH, 'X_images_uint8.npy'), mmap_mode='r')
    X_stats = np.load(os.path.join(BASE_PATH, 'X_stats.npy'))
    y = np.load(os.path.join(BASE_PATH, 'y_labels.npy'))

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_accs, fold_f1s = [], []
    scaler_amp = torch.cuda.amp.GradScaler()

    for fold, (train_val_idx, test_idx) in enumerate(skf.split(X_img, y)):
        print(f"\n🔄 FOLD {fold+1}/{N_FOLDS}...")
        inner_train_idx, inner_val_idx = train_test_split(train_val_idx, test_size=0.1875, stratify=y[train_val_idx], random_state=42)

        s_scaler = StandardScaler()
        S_tr = s_scaler.fit_transform(X_stats[inner_train_idx])
        S_va = s_scaler.transform(X_stats[inner_val_idx])
        S_te = s_scaler.transform(X_stats[test_idx])

        train_l = DataLoader(FusionDataset(X_img[inner_train_idx], S_tr, y[inner_train_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
        val_l = DataLoader(FusionDataset(X_img[inner_val_idx], S_va, y[inner_val_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
        test_l = DataLoader(FusionDataset(X_img[test_idx], S_te, y[test_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

        model = InterFusionModel().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=5e-5)
        criterion = nn.CrossEntropyLoss()
        best_acc = 0.0; best_wts = copy.deepcopy(model.state_dict())

        for ep in range(EPOCHS):
            model.train()
            for i, s, l in tqdm(train_l, desc=f"Ep {ep+1}", leave=False):
                i, s, l = i.to(DEVICE), s.to(DEVICE), l.to(DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    out = model(i, s)
                    loss = criterion(out, l)
                scaler_amp.scale(loss).backward()
                scaler_amp.step(optimizer)
                scaler_amp.update()

            model.eval(); corr=0; tot=0
            with torch.no_grad():
                for i, s, l in val_l:
                    out = model(i.to(DEVICE), s.to(DEVICE))
                    _, p = torch.max(out, 1); corr+=(p==l.to(DEVICE)).sum().item(); tot+=l.size(0)
            if (corr/tot) > best_acc: best_acc=corr/tot; best_wts=copy.deepcopy(model.state_dict())

        model.load_state_dict(best_wts); model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for i, s, l in test_l:
                _, p = torch.max(model(i.to(DEVICE), s.to(DEVICE)), 1)
                preds.extend(p.cpu().numpy()); trues.extend(l.cpu().numpy())

        acc = accuracy_score(trues, preds); f1 = f1_score(trues, preds, average='macro')
        fold_accs.append(acc); fold_f1s.append(f1)
        print(f"   -> Test Acc: {acc:.4f} | F1: {f1:.4f}")
        del model, train_l, val_l, test_l; gc.collect(); torch.cuda.empty_cache()

    print("\n" + "="*85)
    print(f"🏆 {MODEL_NAME} SONUÇ TABLOSU")
    res = {'Metric': ['Accuracy', 'F1-Score'], 'Mean ± Std': [f"{np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}", f"{np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}"]}
    print(pd.DataFrame(res).to_string(index=False))

if __name__ == "__main__":
    run_inter_fusion()

🚀 INTERMEDIATE FUSION BAŞLIYOR...

🔄 FOLD 1/5...


/tmp/ipython-input-311742416.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler()
Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-311742416.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.7998 | F1: 0.7500

🔄 FOLD 2/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-311742416.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8114 | F1: 0.7620

🔄 FOLD 3/5...


...

   -> Test Acc: 0.8181 | F1: 0.7579

🔄 FOLD 4/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-311742416.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8033 | F1: 0.7519

🔄 FOLD 5/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-311742416.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8072 | F1: 0.7585

🏆 Intermediate Fusion SONUÇ TABLOSU
  Metric      Mean ± Std
Accuracy 0.8080 ± 0.0064
F1-Score 0.7561 ± 0.0045


**LATE FUSION**

In [ ]:
# --- LATE FUSION ---
MODEL_NAME = "Late Fusion"

class LateFusionModel(nn.Module):
    def __init__(self, num_classes=9):
        super(LateFusionModel, self).__init__()
        self.img_net = timm.create_model('resnet18', pretrained=True, num_classes=num_classes)
        self.tab_net = nn.Sequential(nn.Linear(27, 64), nn.ReLU(), nn.Linear(64, num_classes))
    def forward(self, i, t):
        return (self.img_net(i) + self.tab_net(t)) / 2

def run_late_fusion():
    print(f"🚀 {MODEL_NAME.upper()} BAŞLIYOR...")
    X_img = np.load(os.path.join(BASE_PATH, 'X_images_uint8.npy'), mmap_mode='r')
    X_stats = np.load(os.path.join(BASE_PATH, 'X_stats.npy'))
    y = np.load(os.path.join(BASE_PATH, 'y_labels.npy'))

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_accs, fold_f1s = [], []
    scaler_amp = torch.cuda.amp.GradScaler()

    for fold, (train_val_idx, test_idx) in enumerate(skf.split(X_img, y)):
        print(f"\n🔄 FOLD {fold+1}/{N_FOLDS}...")
        inner_train_idx, inner_val_idx = train_test_split(train_val_idx, test_size=0.1875, stratify=y[train_val_idx], random_state=42)

        s_scaler = StandardScaler()
        S_tr = s_scaler.fit_transform(X_stats[inner_train_idx])
        S_va = s_scaler.transform(X_stats[inner_val_idx])
        S_te = s_scaler.transform(X_stats[test_idx])

        train_l = DataLoader(FusionDataset(X_img[inner_train_idx], S_tr, y[inner_train_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
        val_l = DataLoader(FusionDataset(X_img[inner_val_idx], S_va, y[inner_val_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
        test_l = DataLoader(FusionDataset(X_img[test_idx], S_te, y[test_idx], get_transforms()), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

        model = LateFusionModel().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.CrossEntropyLoss()
        best_acc = 0.0; best_wts = copy.deepcopy(model.state_dict())

        for ep in range(EPOCHS):
            model.train()
            for i, s, l in tqdm(train_l, desc=f"Ep {ep+1}", leave=False):
                i, s, l = i.to(DEVICE), s.to(DEVICE), l.to(DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    out = model(i, s)
                    loss = criterion(out, l)
                scaler_amp.scale(loss).backward()
                scaler_amp.step(optimizer)
                scaler_amp.update()

            model.eval(); corr=0; tot=0
            with torch.no_grad():
                for i, s, l in val_l:
                    out = model(i.to(DEVICE), s.to(DEVICE))
                    _, p = torch.max(out, 1); corr+=(p==l.to(DEVICE)).sum().item(); tot+=l.size(0)
            if (corr/tot) > best_acc: best_acc=corr/tot; best_wts=copy.deepcopy(model.state_dict())

        model.load_state_dict(best_wts); model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for i, s, l in test_l:
                _, p = torch.max(model(i.to(DEVICE), s.to(DEVICE)), 1)
                preds.extend(p.cpu().numpy()); trues.extend(l.cpu().numpy())

        acc = accuracy_score(trues, preds); f1 = f1_score(trues, preds, average='macro')
        fold_accs.append(acc); fold_f1s.append(f1)
        print(f"   -> Test Acc: {acc:.4f} | F1: {f1:.4f}")
        del model, train_l, val_l, test_l; gc.collect(); torch.cuda.empty_cache()

    print("\n" + "="*85)
    print(f"🏆 {MODEL_NAME} SONUÇ TABLOSU")
    res = {'Metric': ['Accuracy', 'F1-Score'], 'Mean ± Std': [f"{np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}", f"{np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}"]}
    print(pd.DataFrame(res).to_string(index=False))

if __name__ == "__main__":
    run_late_fusion()

🚀 LATE FUSION BAŞLIYOR...

🔄 FOLD 1/5...


/tmp/ipython-input-441072201.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler()
Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-441072201.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8152 | F1: 0.7690

🔄 FOLD 2/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-441072201.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8120 | F1: 0.7724

🔄 FOLD 3/5...


...

   -> Test Acc: 0.8280 | F1: 0.7858

🔄 FOLD 4/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-441072201.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8207 | F1: 0.7755

🔄 FOLD 5/5...


Ep 1:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipython-input-441072201.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   -> Test Acc: 0.8158 | F1: 0.7753

🏆 Late Fusion SONUÇ TABLOSU
  Metric      Mean ± Std
Accuracy 0.8184 ± 0.0056
F1-Score 0.7756 ± 0.0056


**FOOTBALL SCOUT SYSTEM**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import os
import timm
import joblib
from tqdm import tqdm

# --- AYARLAR ---
BASE_PATH = '/content/drive/MyDrive/Deep-Learning'
DATASET_NAME = 'dataset_ready.csv'
BATCH_SIZE = 256
EPOCHS = 12
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 1. DATASET ---
class FusionDataset(Dataset):
    def __init__(self, images, stats, labels, transform=None):
        self.images = images
        self.stats = torch.FloatTensor(stats)
        self.labels = torch.LongTensor(labels)
        self.transform = transform

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img) # (3, 224, 224) yapar
        return img, self.stats[idx], self.labels[idx]

def get_transforms():
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# --- 2. GELİŞMİŞ EARLY FUSION MODELİ ---
class EarlyFusionModel(nn.Module):
    def __init__(self, num_classes=9):
        super(EarlyFusionModel, self).__init__()

        # A. GÖRÜNTÜ (512 Özellik)
        self.img_net = timm.create_model('resnet18', pretrained=True, num_classes=0)

        # B. İSTATİSTİK (256 Özellik - Güçlendirilmiş)
        self.tab_net = nn.Sequential(
            nn.Linear(27, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256), # İstatistik etkisi artırıldı
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

        # C. BİRLEŞTİRME (768 Özellik)
        self.head = nn.Sequential(
            nn.Linear(512 + 256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, i, t):
        f_i = self.img_net(i)
        f_t = self.tab_net(t)

        # Normalizasyon (İki veri türünü eşitlemek için)
        f_i = torch.nn.functional.normalize(f_i, p=2, dim=1)
        f_t = torch.nn.functional.normalize(f_t, p=2, dim=1)

        combined = torch.cat((f_i, f_t), dim=1)
        return self.head(combined)

    def forward_features(self, i, t):
        # Scout DNA'sı (768 Boyutlu Vektör)
        f_i = self.img_net(i)
        f_t = self.tab_net(t)

        f_i = torch.nn.functional.normalize(f_i, p=2, dim=1)
        f_t = torch.nn.functional.normalize(f_t, p=2, dim=1)

        return torch.cat((f_i, f_t), dim=1)

# --- 3. KURULUM FONKSİYONU ---
def build_scout_system():
    print(f"🏭 SCOUT SİSTEMİ KURULUYOR (Enhanced Early Fusion)...")

    # A) Verileri Yükle
    print("📥 Veriler yükleniyor...")
    X_img = np.load(os.path.join(BASE_PATH, 'X_images_uint8.npy'), mmap_mode='r')
    X_stats = np.load(os.path.join(BASE_PATH, 'X_stats.npy'))
    y = np.load(os.path.join(BASE_PATH, 'y_labels.npy'))

    # B) Scaler
    scaler = StandardScaler()
    X_stats_scaled = scaler.fit_transform(X_stats)
    joblib.dump(scaler, os.path.join(BASE_PATH, 'scout_scaler.pkl'))

    # C) Eğitim
    dataset = FusionDataset(X_img, X_stats_scaled, y, transform=get_transforms())
    # Colab'de bazen num_workers=4 hata verebilir, güvenli olması için 2 yaptık
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    model = EarlyFusionModel().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    scaler_amp = torch.amp.GradScaler('cuda') # Mixed Precision Hızlandırıcı

    print("🔥 Model Eğitiliyor...")
    model.train()
    for ep in range(EPOCHS):
        correct = 0; total = 0
        loop = tqdm(loader, desc=f"Epoch {ep+1}/{EPOCHS}", leave=False)

        for i, s, l in loop:
            i, s, l = i.to(DEVICE), s.to(DEVICE), l.to(DEVICE)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                out = model(i, s)
                loss = criterion(out, l)

            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()

            _, preds = torch.max(out, 1)
            correct += (preds == l).sum().item(); total += l.size(0)
            loop.set_postfix(acc=f"{correct/total:.3f}")

        print(f"   Epoch {ep+1} Final Acc: {correct/total:.4f}")

    # Kaydet
    torch.save(model.state_dict(), os.path.join(BASE_PATH, 'scout_model.pth'))
    print("✅ Model Kaydedildi.")

    # D) DNA Çıkarma (Indexing)
    print("🧬 Oyuncu DNA'ları Çıkarılıyor...")
    model.eval()
    seq_loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    embeddings = []

    with torch.no_grad():
        for i, s, _ in tqdm(seq_loader):
            i, s = i.to(DEVICE), s.to(DEVICE)
            feats = model.forward_features(i, s)
            embeddings.append(feats.cpu().numpy())

    np.save(os.path.join(BASE_PATH, 'scout_embeddings.npy'), np.vstack(embeddings))

    # E) Metadata Oluşturma
    csv_path = os.path.join(BASE_PATH, DATASET_NAME)
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        # İsim Formatı: "Kerem Akturkoglu (Turkey - Super Lig)"
        df['scout_name'] = (
            df['player_name'].fillna('?') + " (" +
            df['nationality'].fillna('?') + " - " +
            df['league'].fillna('?') + ")"
        )
        # Sadece saf ismi de saklayalım ki birleştirmede kullanalım
        df['clean_name'] = df['player_name']
        df[['scout_name', 'clean_name', 'position_label']].to_csv(os.path.join(BASE_PATH, 'scout_metadata.csv'), index=False)
    else:
        print("⚠️ CSV bulunamadı!")

    print("\n" + "="*60)
    print("🎉 ADIM 1 TAMAMLANDI: Sistem kuruldu ve dosyalar kaydedildi.")
    print("="*60)

if __name__ == "__main__":
    build_scout_system()

🏭 SCOUT SİSTEMİ KURULUYOR (Enhanced Early Fusion)...
📥 Veriler yükleniyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

🔥 Model Eğitiliyor...


   Epoch 1 Final Acc: 0.2805


...

   Epoch 10 Final Acc: 0.8554


   Epoch 11 Final Acc: 0.8597


   Epoch 12 Final Acc: 0.8685
✅ Model Kaydedildi.
🧬 Oyuncu DNA'ları Çıkarılıyor...


100%|██████████| 244/244 [00:15<00:00, 15.57it/s]



🎉 ADIM 1 TAMAMLANDI: Sistem kuruldu ve dosyalar kaydedildi.


In [ ]:
import numpy as np
import pandas as pd
import os

BASE_PATH = '/content/drive/MyDrive/Deep-Learning'

def merge_player_seasons():
    print("🧹 VERİTABANI TEMİZLENİYOR (Aynı oyuncular birleştiriliyor)...")

    emb_path = os.path.join(BASE_PATH, 'scout_embeddings.npy')
    meta_path = os.path.join(BASE_PATH, 'scout_metadata.csv')

    if not os.path.exists(emb_path): return print("❌ Dosya yok! Adım 1'i çalıştır.")

    embeddings = np.load(emb_path)
    df = pd.read_csv(meta_path)

    print(f"📊 İşlem öncesi: {len(df)} kayıt")

    # Gruplama için sözlük
    unique_players = {}

    # df.itertuples() hızlıdır
    for i, row in enumerate(df.itertuples()):

        # --- DÜZELTME BURADA ---
        # Eğer 'clean_name' sütunu varsa onu kullan, yoksa 'scout_name'den ayıkla
        if hasattr(row, 'clean_name'):
            name = row.clean_name
        else:
            # Örn: "Mauro Icardi (Argentina...)" -> "Mauro Icardi"
            # Parantezden öncesini alıp boşlukları temizliyoruz
            name = str(row.scout_name).split('(')[0].strip()

        if name not in unique_players:
            unique_players[name] = {
                'vectors': [],
                'position': row.position_label,
                'display_name': row.scout_name # Son sezonun bilgisi kalır
            }
        unique_players[name]['vectors'].append(embeddings[i])

    # Ortalamaları al
    new_embeddings = []
    new_display_names = []
    new_positions = []

    print(f"🔄 {len(unique_players)} benzersiz oyuncu işleniyor...")

    for name, data in unique_players.items():
        # Vektörlerin ortalamasını al (Mean Pooling)
        avg_vector = np.mean(np.array(data['vectors']), axis=0)

        # Normalizasyon (Önemli: Ortalama alınca vektör boyu kısalır, tekrar 1 yapmalıyız)
        norm = np.linalg.norm(avg_vector)
        if norm > 0: avg_vector = avg_vector / norm

        new_embeddings.append(avg_vector)
        new_display_names.append(data['display_name'])
        new_positions.append(data['position'])

    # Kaydet
    final_emb = np.vstack(new_embeddings)
    # Metadata'yı güncelliyoruz
    final_df = pd.DataFrame({'scout_name': new_display_names, 'position_label': new_positions})

    np.save(emb_path, final_emb)
    final_df.to_csv(meta_path, index=False)

    print(f"✅ ADIM 2 TAMAMLANDI! Veritabanı boyutu: {len(final_df)} oyuncuya düştü.")
    print(f"📉 Tekrarlar silindi ve Süper Profiller oluşturuldu.")

if __name__ == "__main__":
    merge_player_seasons()

🧹 VERİTABANI TEMİZLENİYOR (Aynı oyuncular birleştiriliyor)...
📊 İşlem öncesi: 5157 kayıt
🔄 5157 benzersiz oyuncu işleniyor...
✅ ADIM 2 TAMAMLANDI! Veritabanı boyutu: 5157 oyuncuya düştü.
📉 Tekrarlar silindi ve Süper Profiller oluşturuldu.


In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics.pairwise import cosine_similarity

BASE_PATH = '/content/drive/MyDrive/Deep-Learning'

class SuperScout:
    def __init__(self):
        print("⚙️ Arama Motoru Başlatılıyor...")
        emb_path = os.path.join(BASE_PATH, 'scout_embeddings.npy')
        meta_path = os.path.join(BASE_PATH, 'scout_metadata.csv')

        if not os.path.exists(emb_path):
            print("❌ Dosyalar bulunamadı!")
            return

        self.embeddings = np.load(emb_path)
        self.meta = pd.read_csv(meta_path)
        print(f"✅ Hazır! {len(self.embeddings)} oyuncu arasından arama yapılıyor.")

    def find_player(self, name_query, top_k=10):
        # İsme göre filtrele
        matches = self.meta[self.meta['scout_name'].str.contains(name_query, case=False, na=False)]

        if len(matches) == 0:
            print(f"❌ '{name_query}' bulunamadı.")
            return

        # İlk eşleşeni hedef al
        target_idx = matches.index[0]
        target_name = matches.iloc[0]['scout_name']
        target_pos = matches.iloc[0]['position_label']

        print(f"\n🔎 ANALİZ: {target_name}")
        print(f"📍 Mevki: {target_pos}")
        print("-" * 50)

        # Benzerlik Hesapla
        target_vec = self.embeddings[target_idx].reshape(1, -1)
        sims = cosine_similarity(target_vec, self.embeddings)[0]

        # Sırala
        idxs = sims.argsort()[::-1][:top_k+1]

        print(f"🏆 EN BENZER {top_k} OYUNCU:")
        count = 0
        for i in idxs:
            # Kendisiyle %100 benzerlik çıkacaktır, onu atla
            if sims[i] > 0.9999: continue

            score = sims[i]
            p_name = self.meta.iloc[i]['scout_name']
            p_pos = self.meta.iloc[i]['position_label']

            # Görsel Bar
            bar = "█" * int(score*20)
            print(f" {count+1}. {p_name}")
            print(f"    {bar} %{score*100:.1f} | {p_pos}")

            count += 1
            if count >= top_k: break

# --- KULLANIM ---
scout = SuperScout()

# Buraya istediğin ismi yazıp çalıştır
scout.find_player("Matteo Guendouzi", top_k=10)

⚙️ Arama Motoru Başlatılıyor...
✅ Hazır! 5157 oyuncu arasından arama yapılıyor.

🔎 ANALİZ: Matteo Guendouzi (France - Serie A)
📍 Mevki: Central Midfield
--------------------------------------------------
🏆 EN BENZER 10 OYUNCU:
 1. Jonas Martin (France - Ligue 1)
    ███████████████████ %97.6 | Defensive Midfield
 2. John Lundstram (England - Premiership)
    ███████████████████ %97.4 | Defensive Midfield
 3. Joan Jordan (Spain - LaLiga)
    ███████████████████ %97.3 | Central Midfield
 4. Sasa Lukic (Serbia - Premier League)
    ███████████████████ %97.1 | Defensive Midfield
 5. Adam Forshaw (England - Premier League)
    ███████████████████ %97.0 | Central Midfield
 6. Jon Moncayola (Spain - LaLiga)
    ███████████████████ %96.9 | Central Midfield
 7. Steeve Beusnard (France - Ligue 2)
    ███████████████████ %96.8 | Central Midfield
 8. Nino Kouter (Slovenia - 1.Lig)
    ███████████████████ %96.8 | Central Midfield
 9. Fabian Ruiz (Spain - Ligue 1)
    ███████████████████ %96.7 | Cen

**FOOTBALL SCOUT SYSTEM-II**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import os
import timm
import joblib
from tqdm import tqdm

# --- AYARLAR ---
BASE_PATH = '/content/drive/MyDrive/Deep-Learning'
DATASET_NAME = 'dataset_ready.csv'
BATCH_SIZE = 256
EPOCHS = 12
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- DATASET & MODEL (Early Fusion - Güçlendirilmiş) ---
class FusionDataset(Dataset):
    def __init__(self, images, stats, labels, transform=None):
        self.images = images
        self.stats = torch.FloatTensor(stats)
        self.labels = torch.LongTensor(labels)
        self.transform = transform
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform: img = self.transform(img)
        return img, self.stats[idx], self.labels[idx]

def get_transforms():
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

class EarlyFusionModel(nn.Module):
    def __init__(self, num_classes=9):
        super(EarlyFusionModel, self).__init__()
        self.img_net = timm.create_model('resnet18', pretrained=True, num_classes=0)
        self.tab_net = nn.Sequential(
            nn.Linear(27, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Linear(512 + 256, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, i, t):
        f_i = torch.nn.functional.normalize(self.img_net(i), p=2, dim=1)
        f_t = torch.nn.functional.normalize(self.tab_net(t), p=2, dim=1)
        return self.head(torch.cat((f_i, f_t), dim=1))
    def forward_features(self, i, t):
        f_i = torch.nn.functional.normalize(self.img_net(i), p=2, dim=1)
        f_t = torch.nn.functional.normalize(self.tab_net(t), p=2, dim=1)
        return torch.cat((f_i, f_t), dim=1)

# --- SİSTEMİ KUR (HAM VERİ) ---
def build_base_system():
    print(f"🏭 ADIM 1: SİSTEM KURULUYOR VE TÜM SEZONLAR ÇIKARILIYOR...")

    # Veri Yükle
    X_img = np.load(os.path.join(BASE_PATH, 'X_images_uint8.npy'), mmap_mode='r')
    X_stats = np.load(os.path.join(BASE_PATH, 'X_stats.npy'))
    y = np.load(os.path.join(BASE_PATH, 'y_labels.npy'))

    scaler = StandardScaler()
    X_stats_scaled = scaler.fit_transform(X_stats)
    joblib.dump(scaler, os.path.join(BASE_PATH, 'scout_scaler.pkl'))

    dataset = FusionDataset(X_img, X_stats_scaled, y, transform=get_transforms())
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    model = EarlyFusionModel().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    scaler_amp = torch.amp.GradScaler('cuda')

    # Eğitim
    print("🔥 Model Eğitiliyor...")
    model.train()
    for ep in range(EPOCHS):
        loop = tqdm(loader, desc=f"Ep {ep+1}/{EPOCHS}", leave=False)
        for i, s, l in loop:
            i, s, l = i.to(DEVICE), s.to(DEVICE), l.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = criterion(model(i, s), l)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()

    torch.save(model.state_dict(), os.path.join(BASE_PATH, 'scout_model.pth'))
    print("✅ Model Kaydedildi.")

    # DNA Çıkarma (Tüm Sezonlar)
    print("🧬 Tüm Sezonların DNA'sı Çıkarılıyor...")
    model.eval()
    seq_loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    embeddings = []

    with torch.no_grad():
        for i, s, _ in tqdm(seq_loader):
            i, s = i.to(DEVICE), s.to(DEVICE)
            embeddings.append(model.forward_features(i, s).cpu().numpy())

    # _all uzantısı ile kaydediyoruz
    np.save(os.path.join(BASE_PATH, 'scout_embeddings_all.npy'), np.vstack(embeddings))

    # Metadata (Sezon Bilgisiyle Birlikte)
    df = pd.read_csv(os.path.join(BASE_PATH, DATASET_NAME))
    # Örn: Mauro Icardi (22/23 - Galatasaray)
    df['scout_name'] = (
        df['player_name'] + " (" +
        df['season'].astype(str) + " - " +
        df['league'] + ")"
    )
    df['pure_name'] = df['player_name'] # Birleştirme için lazım olacak
    df[['scout_name', 'pure_name', 'position_label']].to_csv(os.path.join(BASE_PATH, 'scout_metadata_all.csv'), index=False)

    print("\n✅ ADIM 1 BİTTİ: 'All Seasons' veritabanı hazır.")

if __name__ == "__main__":
    build_base_system()

🏭 ADIM 1: SİSTEM KURULUYOR VE TÜM SEZONLAR ÇIKARILIYOR...
🔥 Model Eğitiliyor...


✅ Model Kaydedildi.
🧬 Tüm Sezonların DNA'sı Çıkarılıyor...


100%|██████████| 244/244 [00:15<00:00, 15.69it/s]



✅ ADIM 1 BİTTİ: 'All Seasons' veritabanı hazır.


In [ ]:
import numpy as np
import pandas as pd
import os

BASE_PATH = '/content/drive/MyDrive/Deep-Learning'

def create_super_profiles():
    print("🏭 ADIM 2: SÜPER PROFİLLER OLUŞTURULUYOR...")

    # Kaynak Dosyalar (All Seasons)
    emb_all = np.load(os.path.join(BASE_PATH, 'scout_embeddings_all.npy'))
    df_all = pd.read_csv(os.path.join(BASE_PATH, 'scout_metadata_all.csv'))

    # Gruplama
    unique_players = {}
    print(f"🔄 {len(df_all)} sezon kaydı taranıyor...")

    for i, row in enumerate(df_all.itertuples()):
        name = row.pure_name # Saf isim (örn: Mauro Icardi)
        if name not in unique_players:
            unique_players[name] = {'vectors': [], 'pos': row.position_label, 'display': row.pure_name}
        unique_players[name]['vectors'].append(emb_all[i])

    # Birleştirme (Average Pooling)
    new_embs = []
    new_names = []
    new_pos = []

    for name, data in unique_players.items():
        # Vektörlerin ortalamasını al
        avg_vec = np.mean(np.array(data['vectors']), axis=0)
        # Normalize et
        norm = np.linalg.norm(avg_vec)
        if norm > 0: avg_vec = avg_vec / norm

        new_embs.append(avg_vec)
        new_names.append(data['display']) # Sadece isim (örn: Mauro Icardi)
        new_pos.append(data['pos'])

    # Kaydet (_merged uzantısı ile)
    np.save(os.path.join(BASE_PATH, 'scout_embeddings_merged.npy'), np.vstack(new_embs))
    pd.DataFrame({'scout_name': new_names, 'position_label': new_pos}).to_csv(os.path.join(BASE_PATH, 'scout_metadata_merged.csv'), index=False)

    print(f"✅ ADIM 2 BİTTİ: {len(new_embs)} adet Süper Profil oluşturuldu.")

if __name__ == "__main__":
    create_super_profiles()

🏭 ADIM 2: SÜPER PROFİLLER OLUŞTURULUYOR...
🔄 15585 sezon kaydı taranıyor...
✅ ADIM 2 BİTTİ: 5157 adet Süper Profil oluşturuldu.


In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics.pairwise import cosine_similarity

BASE_PATH = '/content/drive/MyDrive/Deep-Learning'

class SuperScout:
    def __init__(self, mode='merged'):
        """
        mode: 'merged' -> Süper Profil (Oyuncu bazlı, tek sonuç)
        mode: 'all'    -> Tüm Sezonlar (Sezon bazlı, çoklu sonuç)
        """
        self.mode = mode
        print(f"⚙️ Sistem Yükleniyor (Mod: {mode.upper()})...")

        suffix = "_merged" if mode == 'merged' else "_all"
        emb_path = os.path.join(BASE_PATH, f'scout_embeddings{suffix}.npy')
        meta_path = os.path.join(BASE_PATH, f'scout_metadata{suffix}.csv')

        if not os.path.exists(emb_path):
            print("❌ Dosyalar bulunamadı! Önce kurulum adımlarını yapın.")
            return

        self.embeddings = np.load(emb_path)
        self.meta = pd.read_csv(meta_path)
        print(f"✅ Hazır! {len(self.embeddings)} kayıt taramaya hazır.")

    def find_player(self, name_query, top_k=10):
        # İsme göre filtrele
        matches = self.meta[self.meta['scout_name'].str.contains(name_query, case=False, na=False)]

        if len(matches) == 0:
            print(f"❌ '{name_query}' bulunamadı.")
            return

        # İlk eşleşeni hedef al
        target_idx = matches.index[0]
        target_name = matches.iloc[0]['scout_name']
        target_pos = matches.iloc[0]['position_label']

        print(f"\n🔎 ANALİZ: {target_name}")
        print(f"📍 Mevki: {target_pos}")
        print("-" * 60)

        # Benzerlik Hesapla
        target_vec = self.embeddings[target_idx].reshape(1, -1)
        sims = cosine_similarity(target_vec, self.embeddings)[0]

        # Sırala
        idxs = sims.argsort()[::-1][:top_k+1]

        print(f"🏆 EN BENZER {top_k} OYUNCU ({self.mode.upper()} MODU):")
        count = 0
        for i in idxs:
            if sims[i] > 0.99999: continue # Kendisi

            # Eğer 'Merged' modundaysak ve aynı isim çıktıysa atla (Nadiren olabilir)
            p_name = self.meta.iloc[i]['scout_name']
            if self.mode == 'merged' and p_name == target_name: continue

            score = sims[i]
            p_pos = self.meta.iloc[i]['position_label']

            bar = "█" * int(score*20)
            print(f" {count+1}. {p_name}")
            print(f"    {bar} %{score*100:.1f} | {p_pos}")

            count += 1
            if count >= top_k: break

# --- KULLANIM ÖRNEKLERİ ---

print("\n--- TEST 1: SÜPER PROFİL MODU (Genel Benzerlik) ---")
scout_merged = SuperScout(mode='merged')
scout_merged.find_player("Cristiano Ronaldo")

print("\n\n--- TEST 2: ALL SEASONS MODU (Sezonluk Benzerlik) ---")
scout_all = SuperScout(mode='all')
scout_all.find_player("Cristiano Ronaldo") # Hangi sezonu ilk bulursa ona göre arar


--- TEST 1: SÜPER PROFİL MODU (Genel Benzerlik) ---
⚙️ Sistem Yükleniyor (Mod: MERGED)...
✅ Hazır! 5157 kayıt taramaya hazır.

🔎 ANALİZ: Cristiano Ronaldo
📍 Mevki: Centre-Forward
------------------------------------------------------------
🏆 EN BENZER 10 OYUNCU (MERGED MODU):
 1. Cyle Larin
    ███████████████████ %97.5 | Centre-Forward
 2. Pierre Emerick Aubameyang
    ███████████████████ %97.4 | Centre-Forward
 3. Mbaye Niang
    ███████████████████ %96.9 | Centre-Forward
 4. Marcus Thuram
    ███████████████████ %96.7 | Centre-Forward
 5. Abdallah Sima
    ███████████████████ %96.0 | Left Winger
 6. Cenk Tosun
    ███████████████████ %95.7 | Centre-Forward
 7. Benjamin Sesko
    ███████████████████ %95.4 | Centre-Forward
 8. Mike Van Duinen
    ███████████████████ %95.3 | Centre-Forward
 9. Bryan Linssen
    ███████████████████ %95.2 | Centre-Forward
 10. Ollie Watkins
    ███████████████████ %95.2 | Centre-Forward


--- TEST 2: ALL SEASONS MODU (Sezonluk Benzerlik) ---
⚙️ Sistem Y